# 03 — Data Quality Assessment

**Sprint 4: Data Inspection & Exploratory Data Analysis (EDA)**

## Objective
Assess the dataset's quality before analysis or modelling. This notebook investigates missing values, duplicate records, invalid or unexpected values, empty strings, null-like placeholders, and data types.

## Dataset context
The reference notebook uses the **Telco Customer Churn** dataset. One row is expected to represent a customer. Checks that depend on known Telco columns are performed only when those columns are present; other checks are dataset-wide.

## Important approach
- Begin by measuring and documenting quality issues.
- Do not silently delete rows, replace values, or change the original DataFrame during assessment.
- Treat automated flags as candidates for review: whether a value is invalid ultimately depends on the dataset's definitions and business rules.

## Setup — Load the dataset

### Concept
Load the same CSV used in **`02_Data_Inspection.ipynb`** so the quality assessment follows directly from the inspection notebook.

### Example
A customer record may have a blank `TotalCharges` value, an unexpected category, or a repeated `customerID`. The first step is to load the data and preserve an untouched copy for comparison.

### Code

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path("../dataset/WA_Fn-UseC_-Telco-Customer-Churn.csv")

# Also support running this notebook from the project root.
if not DATA_PATH.exists():
    alternative = Path("dataset/WA_Fn-UseC_-Telco-Customer-Churn.csv")
    if alternative.exists():
        DATA_PATH = alternative

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"CSV not found: {DATA_PATH.resolve()}. "
        "Download the dataset and update DATA_PATH."
    )

df = pd.read_csv(DATA_PATH)
df_original = df.copy(deep=True)

print("Dataset loaded successfully.")
print("Shape:", df.shape)
display(df.head())

Dataset loaded successfully.
Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### Output explanation
The output confirms the CSV was loaded and displays its dimensions and first records.

### Insight
Keep `df_original` unchanged throughout this notebook. The checks below report possible problems; they do not automatically clean the data.

## 1. Missing Values — Count and Percentage

### Concept
Pandas treats values such as `NaN` and `None` as missing. Count missing cells in each column and calculate the percentage of rows affected.

### Example
If 70 out of 1,000 values in `TotalCharges` are missing, its missing percentage is 7%.

### Code

In [5]:
missing_report = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": (df.isna().mean() * 100).round(2)
}).sort_values("missing_count", ascending=False)

display(missing_report)
print("Total missing cells:", int(df.isna().sum().sum()))
print("Columns with missing values:", int(df.isna().any().sum()))

,missing_count,missing_percentage
customerID,0,0.0
gender,0,0.0
SeniorCitizen,0,0.0
Partner,0,0.0
Dependents,0,0.0
tenure,0,0.0
PhoneService,0,0.0
MultipleLines,0,0.0
InternetService,0,0.0
OnlineSecurity,0,0.0


Total missing cells: 0
Columns with missing values: 0


### Output explanation
Each column has a missing-cell count and a percentage. The report is sorted so columns with the most missing values appear first.

### Insight
A high missing percentage may affect analysis or model performance. Investigate why values are missing before deciding whether to remove or impute them.

## 2. Missing Values by Column — Visual Summary

### Concept
A bar chart makes it easier to compare missing-value counts across columns.

### Code

In [8]:
import matplotlib.pyplot as plt

missing_counts = df.isna().sum()
missing_counts = missing_counts[missing_counts > 0].sort_values(ascending=False)

if missing_counts.empty:
    print("No pandas-recognized missing values found.")
else:
    missing_counts.plot(kind="bar", figsize=(10, 4))
    plt.title("Missing Values by Column")
    plt.xlabel("Column")
    plt.ylabel("Number of missing values")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

No pandas-recognized missing values found.


### Output explanation
Only columns with at least one pandas-recognized missing value are shown. If no bars appear, the printed message indicates that no such values were detected.

### Insight
This chart is a quick comparison, while the table in Section 1 provides exact counts and percentages.

## 3. Missing-Value Patterns

### Concept
A missingness pattern shows which columns are missing together in the same records. Grouping rows by their missing-column pattern can reveal systematic gaps.

### Code

In [9]:
missing_mask = df.isna()
pattern_labels = missing_mask.apply(
    lambda row: ", ".join(row.index[row].tolist()) if row.any() else "No missing values",
    axis=1
)

pattern_summary = (
    pattern_labels.value_counts()
    .rename_axis("missing_columns_pattern")
    .reset_index(name="record_count")
)
pattern_summary["record_percentage"] = (
    pattern_summary["record_count"] / len(df) * 100
).round(2) if len(df) else 0

display(pattern_summary.head(20))
print("Unique missingness patterns:", pattern_labels.nunique())

,missing_columns_pattern,record_count,record_percentage
0,No missing values,7043,100.0


Unique missingness patterns: 1


### Output explanation
Each row describes a pattern of missing columns and how often that pattern occurs. `No missing values` means that record has no pandas-recognized missing cells.

### Insight
If the same fields are frequently missing together, the missingness may be related to a shared collection process or a conditional field.

## 4. Empty Strings and Whitespace-Only Values

### Concept
An empty string (`""`) is not automatically considered missing by `isna()`. Cells containing only spaces or tabs also need separate inspection.

### Code

In [10]:
string_columns = df.select_dtypes(include=["object", "string"]).columns
empty_string_counts = {}

for column in string_columns:
    values = df[column].astype("string")
    empty_string_counts[column] = int(values.str.fullmatch(r"\s*", na=False).sum())

empty_string_report = (
    pd.Series(empty_string_counts, name="empty_or_whitespace_count")
    .to_frame()
    .sort_values("empty_or_whitespace_count", ascending=False)
)
display(empty_string_report)

,empty_or_whitespace_count
TotalCharges,11
customerID,0
gender,0
Partner,0
PhoneService,0
Dependents,0
InternetService,0
OnlineSecurity,0
OnlineBackup,0
MultipleLines,0


### Output explanation
The report counts blank and whitespace-only text cells for each text column. A value that is already missing is not counted as whitespace-only.

### Insight
Blank text values can be overlooked in ordinary missing-value counts. Review these separately before deciding whether they should be treated as missing.

## 5. Null-Like Text Placeholders

### Concept
Files sometimes use text such as `NA`, `N/A`, `null`, `none`, or `-` to represent missing data. These are ordinary strings unless handled during import or explicitly identified.

### Code

In [11]:
null_like_tokens = {"na", "n/a", "null", "none", "nan", "missing", "-", "--"}

null_like_rows = []
for column in string_columns:
    values = df[column].astype("string").str.strip().str.casefold()
    mask = values.isin(null_like_tokens)
    count = int(mask.sum())
    if count:
        examples = df.loc[mask, column].astype("string").value_counts().head(10).to_dict()
        null_like_rows.append({
            "column": column,
            "null_like_count": count,
            "examples": examples
        })

null_like_report = pd.DataFrame(
    null_like_rows,
    columns=["column", "null_like_count", "examples"]
)
if null_like_report.empty:
    print("No configured null-like text placeholders found.")
else:
    display(null_like_report)

No configured null-like text placeholders found.


### Output explanation
Only the listed tokens are checked, without changing the original data. The report includes example spellings found in each affected column.

### Insight
The token list is configurable and may need adjustment to the dataset's documentation. For example, a hyphen can be a legitimate category in some datasets.

## 6. Duplicate Records — Completely Duplicated Rows

### Concept
A completely duplicated row matches another row across every column. `keep=False` marks every row that belongs to a duplicate group.

### Code

In [12]:
duplicate_all_mask = df.duplicated(keep=False)
duplicate_all_rows = df.loc[duplicate_all_mask].copy()

print("Rows that belong to a complete-duplicate group:", int(duplicate_all_mask.sum()))
print("Number of extra duplicate rows (keep='first'):", int(df.duplicated().sum()))
display(duplicate_all_rows.head(20))

Rows that belong to a complete-duplicate group: 0
Number of extra duplicate rows (keep='first'): 0


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn


### Output explanation
The first count includes all members of duplicate groups. The second count is the number of repeated rows beyond the first occurrence of each group.

### Insight
A fully duplicated row may be a data-entry or ingestion issue, but confirm the expected grain of the dataset before removing anything.

## 7. Duplicate Records — Identifier and Potential Duplicates

### Concept
Records can be duplicates even when some fields differ. If the dataset has a customer identifier, repeated identifier values are useful candidates for review.

### Code

In [13]:
identifier_candidates = [
    column for column in df.columns
    if column.strip().casefold() in {"customerid", "customer_id", "id", "record_id"}
]

if identifier_candidates:
    for column in identifier_candidates:
        id_mask = df[column].notna() & df[column].duplicated(keep=False)
        repeated_ids = df.loc[id_mask].sort_values(column)
        print(f"{column}: {int(id_mask.sum())} rows have a repeated non-missing identifier")
        display(repeated_ids.head(20))
else:
    print("No common identifier column name was found. Review the column list and define the correct business key.")

# Potential duplicate candidates: exact match across all available columns except common identifiers.
non_id_columns = [c for c in df.columns if c not in identifier_candidates]
if non_id_columns:
    potential_mask = df.duplicated(subset=non_id_columns, keep=False)
    potential_duplicates = df.loc[potential_mask].copy()
    print("Rows matching on non-identifier columns:", int(potential_mask.sum()))
    display(potential_duplicates.head(20))
else:
    print("No non-identifier columns available for a potential-duplicate check.")

customerID: 0 rows have a repeated non-missing identifier


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn


Rows matching on non-identifier columns: 42


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
22,1066-JKSGK,Male,0,No,No,1,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Mailed check,20.15,20.15,Yes
100,6380-ARCEH,Male,0,No,No,1,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Mailed check,20.20,20.2,No
542,2866-IKBTM,Female,0,No,No,1,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Mailed check,19.55,19.55,No
646,0887-HJGAR,Male,0,No,No,1,Yes,No,DSL,No,...,No,No,No,No,Month-to-month,Yes,Mailed check,45.70,45.7,Yes
662,3030-ZKIWL,Male,0,No,No,1,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Mailed check,20.05,20.05,No
690,8262-COGGB,Male,0,No,No,1,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Mailed check,20.45,20.45,No
964,9117-SHLZX,Male,0,No,No,1,Yes,No,DSL,No,...,No,No,No,No,Month-to-month,Yes,Mailed check,45.70,45.7,Yes
976,5875-YPQFJ,Male,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,69.90,69.9,Yes
1243,5552-ZNFSJ,Male,0,No,No,1,Yes,No,DSL,No,...,No,No,No,No,Month-to-month,No,Electronic check,45.30,45.3,Yes
1338,1934-SJVJK,Male,0,No,No,1,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Mailed check,20.15,20.15,Yes


### Output explanation
Repeated identifier values are listed where a common ID column exists. The second check compares rows using all columns except recognized identifiers; it is a candidate check, not proof of duplicate customers.

### Insight
For the Telco dataset, `customerID` is expected to identify a customer. Repeated IDs should be reviewed, not automatically deleted. In other datasets, choose a business key based on the data dictionary.

## 8. Invalid Values — Negative Numbers Where Inappropriate

### Concept
Negative numbers can be valid for some measures, such as profit or temperature, but invalid for non-negative quantities such as tenure, charges, counts, or area. Define the rule from the meaning of each column.

### Code

In [14]:
# Known Telco fields that are expected to be non-negative.
known_non_negative = {
    "tenure", "MonthlyCharges", "TotalCharges",
    "SeniorCitizen", "num_dependents", "dependents"
}
negative_value_rows = []

for column in df.columns:
    numeric_values = pd.to_numeric(df[column], errors="coerce")
    if column in known_non_negative or (
        pd.api.types.is_numeric_dtype(df[column]) and
        any(token in column.casefold() for token in ["count", "charge", "amount", "quantity", "tenure", "age"])
    ):
        mask = numeric_values < 0
        if mask.any():
            negative_value_rows.append({
                "column": column,
                "negative_count": int(mask.sum()),
                "example_values": numeric_values[mask].head(10).tolist()
            })

negative_report = pd.DataFrame(
    negative_value_rows,
    columns=["column", "negative_count", "example_values"]
)
if negative_report.empty:
    print("No negative values found in the configured non-negative fields.")
else:
    display(negative_report)

No negative values found in the configured non-negative fields.


### Output explanation
The report lists negative values in fields that are expected to be non-negative according to the configured rules.

### Insight
This is a rule-based screen, not a universal rule for every numeric column. Review units and definitions before labelling any negative value as invalid.

## 9. Impossible or Out-of-Range Values

### Concept
A value may be numeric and non-missing but still be impossible or outside an expected range. Rules must be based on documented field meanings.

### Code

In [15]:
range_rules = {
    # Telco Customer Churn dataset: documented-style expected ranges.
    "tenure": (0, 100),            # months; broad upper bound for screening
    "MonthlyCharges": (0, None),   # charges cannot be negative
    "TotalCharges": (0, None),    # charges cannot be negative
    "SeniorCitizen": (0, 1),       # binary indicator in this dataset
}

range_report_rows = []
for column, (lower, upper) in range_rules.items():
    if column not in df.columns:
        continue
    numeric_values = pd.to_numeric(df[column], errors="coerce")
    mask = numeric_values.notna()
    if lower is not None:
        mask &= numeric_values < lower
    if upper is not None:
        mask |= numeric_values.notna() & (numeric_values > upper)
    invalid_values = numeric_values[mask]
    range_report_rows.append({
        "column": column,
        "lower_bound": lower,
        "upper_bound": upper,
        "out_of_range_count": int(mask.sum()),
        "examples": invalid_values.head(10).tolist()
    })

range_report = pd.DataFrame(range_report_rows)
if range_report.empty:
    print("None of the configured range-rule columns were found.")
else:
    display(range_report)

,column,lower_bound,upper_bound,out_of_range_count,examples
0,tenure,0,100.0,0,[]
1,MonthlyCharges,0,NaN,0,[]
2,TotalCharges,0,NaN,0,[]
3,SeniorCitizen,0,1.0,0,[]


### Output explanation
The report shows which configured columns were checked, their screening bounds, the number of values outside those bounds, and examples.

### Insight
The bounds are explicit assumptions for screening and should be verified against the dataset documentation. Do not apply generic limits to fields whose units or valid range are unknown.

## 10. Incorrect Categories and Unexpected Values

### Concept
Categorical fields should contain values from an expected set. Inconsistent spelling, capitalization, or unexpected labels can split one category into several groups.

### Code

In [16]:
expected_categories = {
    "gender": {"female", "male"},
    "Partner": {"yes", "no"},
    "Dependents": {"yes", "no"},
    "PhoneService": {"yes", "no"},
    "MultipleLines": {"yes", "no", "no phone service"},
    "InternetService": {"dsl", "fiber optic", "no"},
    "OnlineSecurity": {"yes", "no", "no internet service"},
    "OnlineBackup": {"yes", "no", "no internet service"},
    "DeviceProtection": {"yes", "no", "no internet service"},
    "TechSupport": {"yes", "no", "no internet service"},
    "StreamingTV": {"yes", "no", "no internet service"},
    "StreamingMovies": {"yes", "no", "no internet service"},
    "Contract": {"month-to-month", "one year", "two year"},
    "PaperlessBilling": {"yes", "no"},
    "PaymentMethod": {
        "electronic check", "mailed check",
        "bank transfer (automatic)", "credit card (automatic)"
    },
    "Churn": {"yes", "no"}
}

category_issue_rows = []
for column, allowed_values in expected_categories.items():
    if column not in df.columns:
        continue
    observed = df[column].dropna().astype("string").str.strip().str.casefold()
    unexpected_mask = ~observed.isin(allowed_values)
    unexpected_values = observed[unexpected_mask].value_counts()
    if not unexpected_values.empty:
        category_issue_rows.append({
            "column": column,
            "unexpected_value_count": int(unexpected_values.sum()),
            "unexpected_distinct_values": unexpected_values.to_dict()
        })

category_issue_report = pd.DataFrame(
    category_issue_rows,
    columns=["column", "unexpected_value_count", "unexpected_distinct_values"]
)
if category_issue_report.empty:
    print("No unexpected values found in the configured categorical fields.")
else:
    display(category_issue_report)

No unexpected values found in the configured categorical fields.


### Output explanation
Only configured columns are checked. Values are trimmed and compared without case sensitivity, but the original values remain unchanged. Unexpected values are shown with their frequencies.

### Insight
The accepted categories reflect the Telco dataset. If the source file uses a different schema or category definitions, update `expected_categories` rather than assuming every unlisted value is wrong.

## 11. Invalid Dates

### Concept
Date fields may contain values that cannot be parsed as dates. First identify likely date columns by name, then test whether non-missing values can be parsed.

### Code

In [17]:
date_name_candidates = [
    column for column in df.columns
    if any(token in column.casefold() for token in ["date", "datetime", "timestamp"])
]

if not date_name_candidates:
    print("No date-like column names found. If a date column has another name, add it to date_name_candidates.")
else:
    date_issue_rows = []
    for column in date_name_candidates:
        raw = df[column]
        non_missing = raw.notna() & raw.astype("string").str.strip().ne("")
        parsed = pd.to_datetime(raw[non_missing], errors="coerce")
        invalid_mask = parsed.isna()
        invalid_examples = raw[non_missing][invalid_mask].astype("string").head(10).tolist()
        date_issue_rows.append({
            "column": column,
            "non_empty_values_checked": int(non_missing.sum()),
            "unparseable_date_count": int(invalid_mask.sum()),
            "invalid_examples": invalid_examples
        })
    display(pd.DataFrame(date_issue_rows))

No date-like column names found. If a date column has another name, add it to date_name_candidates.


### Output explanation
Columns whose names contain `date`, `datetime`, or `timestamp` are checked. The report counts non-empty values that could not be parsed.

### Insight
The Telco reference dataset may not contain date columns. If dates are stored under another name, explicitly add the column to the candidate list. Parsing success alone does not prove a date is logically valid.

## 12. Incorrect Data Types and Numeric Conversion Issues

### Concept
A column can contain numbers stored as text. Attempting a numeric conversion on a copy helps identify non-empty values that cannot be interpreted as numbers.

### Code

In [18]:
dtype_report = pd.DataFrame({
    "stored_dtype": df.dtypes.astype(str),
    "non_missing_count": df.notna().sum(),
    "distinct_non_missing_values": df.nunique(dropna=True)
})
display(dtype_report)

numeric_name_candidates = [
    column for column in df.columns
    if any(token in column.casefold() for token in [
        "tenure", "charge", "amount", "price", "count", "quantity", "age", "year", "rainfall"
    ])
]

conversion_rows = []
for column in numeric_name_candidates:
    raw = df[column]
    non_empty = raw.notna() & raw.astype("string").str.strip().ne("")
    converted = pd.to_numeric(raw, errors="coerce")
    invalid_mask = non_empty & converted.isna()
    conversion_rows.append({
        "column": column,
        "non_empty_values_checked": int(non_empty.sum()),
        "not_convertible_to_numeric": int(invalid_mask.sum()),
        "examples": raw[invalid_mask].astype("string").head(10).tolist()
    })

if conversion_rows:
    display(pd.DataFrame(conversion_rows))
else:
    print("No columns matched the numeric-name screening rules.")

,stored_dtype,non_missing_count,distinct_non_missing_values
customerID,str,7043,7043
gender,str,7043,2
SeniorCitizen,int64,7043,2
Partner,str,7043,2
Dependents,str,7043,2
tenure,int64,7043,73
PhoneService,str,7043,2
MultipleLines,str,7043,3
InternetService,str,7043,3
OnlineSecurity,str,7043,3


,column,non_empty_values_checked,not_convertible_to_numeric,examples
0,tenure,7043,0,[]
1,MonthlyCharges,7043,0,[]
2,TotalCharges,7032,0,[]


### Output explanation
The first table shows stored data types and basic column counts. The second table checks likely numeric fields for non-empty values that fail numeric conversion.

### Insight
A conversion failure can reveal values such as text labels in numeric fields. Column names are only a heuristic; use the data dictionary to decide which columns should be numeric.

## 13. Unexpected Values — Frequency Review

### Concept
Unexpected values may not violate an explicit rule but can still deserve investigation. Reviewing distinct values and their frequencies helps find spelling variations, rare labels, or unusual numeric values.

### Code

In [19]:
for column in df.select_dtypes(include=["object", "string", "category"]).columns:
    print(f"\n--- {column} ---")
    display(df[column].value_counts(dropna=False).head(20).to_frame("count"))

numeric_columns = df.select_dtypes(include=np.number).columns
if len(numeric_columns):
    print("\nNumeric summary for range and outlier review:")
    display(df[numeric_columns].describe().T)


--- customerID ---


,count
customerID,
7590-VHVEG,1
5575-GNVDE,1
3668-QPYBK,1
7795-CFOCW,1
9237-HQITU,1
9305-CDSKC,1
1452-KIOVK,1
6713-OKOMC,1
7892-POOKP,1



--- gender ---


,count
gender,
Male,3555
Female,3488



--- Partner ---


...

,count
TotalCharges,
20.2,11
,11
19.75,9
19.9,8
20.05,8
19.65,8
45.3,7
19.55,7
20.15,6



--- Churn ---


,count
Churn,
No,5174
Yes,1869



Numeric summary for range and outlier review:


,count,mean,std,min,25%,50%,75%,max
SeniorCitizen,7043.0,0.162147,0.368612,0.00,0.0,0.00,0.00,1.00
tenure,7043.0,32.371149,24.559481,0.00,9.0,29.00,55.00,72.00
MonthlyCharges,7043.0,64.761692,30.090047,18.25,35.5,70.35,89.85,118.75


### Output explanation
Categorical columns show up to 20 most frequent values, including missing values. Numeric columns show descriptive statistics such as minimum, maximum, quartiles, and mean.

### Insight
Rare values are not automatically errors. Compare unusual observations with the source documentation and domain context before deciding how to handle them.

## 14. Consolidated Data Quality Summary

### Concept
A summary table gathers the main issue counts into one place. It is useful for recording a baseline before any cleaning is performed.

### Code

In [20]:
quality_summary = pd.DataFrame([
    {"check": "Missing cells recognized by pandas", "issue_count": int(df.isna().sum().sum())},
    {"check": "Columns with pandas-recognized missing values", "issue_count": int(df.isna().any().sum())},
    {"check": "Empty or whitespace-only text cells", "issue_count": int(empty_string_report["empty_or_whitespace_count"].sum()) if not empty_string_report.empty else 0},
    {"check": "Configured null-like text placeholders", "issue_count": int(null_like_report["null_like_count"].sum()) if not null_like_report.empty else 0},
    {"check": "Extra complete duplicate rows", "issue_count": int(df.duplicated().sum())},
    {"check": "Rows in complete-duplicate groups", "issue_count": int(duplicate_all_mask.sum())},
    {"check": "Negative values flagged by configured rules", "issue_count": int(negative_report["negative_count"].sum()) if not negative_report.empty else 0},
    {"check": "Out-of-range values flagged by configured rules", "issue_count": int(range_report["out_of_range_count"].sum()) if not range_report.empty else 0},
    {"check": "Unexpected category values in configured columns", "issue_count": int(category_issue_report["unexpected_value_count"].sum()) if not category_issue_report.empty else 0},
])

display(quality_summary)
print("Rows assessed:", len(df))
print("Columns assessed:", len(df.columns))

,check,issue_count
0,Missing cells recognized by pandas,0
1,Columns with pandas-recognized missing values,0
2,Empty or whitespace-only text cells,11
3,Configured null-like text placeholders,0
4,Extra complete duplicate rows,0
5,Rows in complete-duplicate groups,0
6,Negative values flagged by configured rules,0
7,Out-of-range values flagged by configured rules,0
8,Unexpected category values in configured columns,0


Rows assessed: 7043
Columns assessed: 21


### Output explanation
The table reports the total number of flags found by each check. Different checks can flag the same cell or row, so these counts should not be added together as if they were unique problems.

### Insight
Use this summary as a triage list. Inspect individual reports to understand which records and values caused each flag before making cleaning decisions.